# NeuroBot: export of real EEG signal for the replay logs (no training)

**What this notebook does.** It adds *real, full-rate EEG* to the replay logs that the web console plays back:

* for every replay window, the stored trace of the displayed channels at the **full sampling rate** (160 samples for the limb tasks, 100 for the finger task) instead of the 40-sample decimated trace;
* the **relative band power** of every displayed channel (theta 4-8 Hz, mu 8-13 Hz, beta 13-30 Hz, low gamma 30-40 Hz), computed from those full-rate windows;
* for the finger replays, which currently store no EEG at all, real traces and band power as well.

**What it does not do.** It does **not train anything**. The models, the ONNX files and every accuracy number stay exactly as they are.

**How it stays correct.** It rebuilds the same held-out windows with the same code and split as the export notebooks, and then checks the result against your existing replay logs before it writes anything:

1. the number of windows is identical;
2. the trial id of every window is identical;
3. the true class of every window is identical;
4. if the ONNX files are available, the exported model gives the **same predictions** on the rebuilt windows as the stored log (agreement of at least 99.9 %).

If any check fails for a log, **that log is not written** and the reason is printed. Nothing is attached to a window that cannot be proven to be the same window.

**What you need to attach as input** (Add data, then Upload, then New dataset) one dataset that contains these files from your repository: `backend/data/replay_*.json` and `backend/checkpoints/*.onnx` plus `backend/checkpoints/finger/*.onnx`. The notebook finds them by name anywhere below `/kaggle/input`. The limb part downloads two PhysioNet subjects (a few minutes). The finger part downloads subject S01 (16 GB, as in the export notebook) and needs Internet on.

**Result:** `neurobot_signal_logs.zip` in the Output tab. Copy its `replay_*.json` files over `backend/data/` and redeploy.

In [ ]:
import os, re, glob, json, copy, time, shutil, zipfile, tarfile, warnings
warnings.filterwarnings("ignore")
import numpy as np
import scipy.io, scipy.signal
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ------------------------------ configuration ------------------------------
DRY_RUN       = False           # False = the REAL run (full epochs, early stopping). True = quick 3-epoch smoke test (writes ..._DRYRUN.json).
SUBJECTS      = [1]             # 1..21; one subject per Kaggle version (each is a 16-24 GB download plus hours of training)
NCLASS_LIST   = [2, 3]          # 4 is not supported (label 3 exists only in the offline session)
MODES         = ["ME", "MI"]
DATA_ROOT     = None            # or e.g. "/kaggle/input/finger-bci" if you attached data containing S01/, S02/, ...
FIGSHARE_ID   = 29104040
SUBJECT_FILE_MAP = {}           # optional {subject_id: ["exact_file_name.zip", ...]} if auto-matching fails

TMP_DIR   = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
WORK_DIR  = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
CACHE_DIR = os.path.join(WORK_DIR, "finger_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

PARAMS = dict(maxtriallen=5, windowlen=1.0, step_sec=0.125, downsrate=100, bandpass=(4, 40))
ZERO_PHASE = False              # official code uses a causal lfilter per window; True = filtfilt
FT_FRAC = 0.30                  # share of the held-out session's first trials used for fine-tuning
TWO_CLASS_FOLDERS = "all"       # "all" (as the official code) or "2class" (only the 2class online folders)
MODELS_TO_RUN = ["tcformer"]       # EXPORT notebook: TCFormer only
SEEDS = {"tcformer": [0], "eegnet": [0]}   # EXPORT notebook: one seed (seed 0)
EXT_SUBJECTS = []               # optional: e.g. [2, 3] -> offline windows of those subjects used for cross-subject pretraining
EXT_STRIDE = 4                  # keep every 4th window of external subjects (memory)
EXT_EPOCHS, EXT_PATIENCE = 30, 5
EPOCH_OVERRIDE = 25               # caps BOTH stages at 25 epochs (no early stopping): the practical run. None = full limits (150 pretrain / 80 fine-tune), which took over 2 h for the first configuration alone and had not finished.
MAX_RUNTIME_HOURS = 10.5        # Kaggle kills a session at 12 h and then keeps NO output; stop starting new work before that
MIN_BLOCK_MINUTES = 60          # do not start a new class configuration / model block with less than this left
PRIOR_RESULTS_PATHS = []        # e.g. ["/kaggle/input/my-finger-run-1/finger_results_export.json"] to continue an earlier version
T0 = time.time()
if DRY_RUN:
    SUBJECTS, NCLASS_LIST = SUBJECTS[:1], [2, 3]
    SEEDS = {"tcformer": [0], "eegnet": [0]}
    EXT_SUBJECTS, EXT_EPOCHS, EXT_PATIENCE, EPOCH_OVERRIDE = [], 2, 2, 3
VARIANTS = ["baseline"] + (["cross"] if EXT_SUBJECTS else [])
CLASS_SETS = {2: [1, 4], 3: [1, 2, 4]}      # label ids used by the official code: 1 thumb, 2 index, 4 pinky
assert set(NCLASS_LIST) <= {2, 3}, "only 2- and 3-class are supported (label 3 exists only in the offline session)"
assert not set(EXT_SUBJECTS) & set(SUBJECTS), "external subjects must never include evaluated subjects"
RESULTS_PATH = os.path.join(WORK_DIR, "finger_results_v2_DRYRUN.json" if DRY_RUN else "finger_results_export.json")

def time_left():
    return MAX_RUNTIME_HOURS * 3600 - (time.time() - T0)

def _json_default(o):
    if hasattr(o, "item"): return o.item()          # numpy scalars
    if hasattr(o, "tolist"): return o.tolist()      # numpy arrays
    return str(o)

def load_json_safe(path):
    """{} for a missing, empty (0-byte) or corrupt file instead of crashing."""
    try:
        if os.path.exists(path) and os.path.getsize(path) > 0:
            with open(path) as fh:
                return json.load(fh)
    except Exception as e:
        print(f"  could not read {path} ({type(e).__name__}: {e}) - ignoring it")
    return {}

def load_results_merged():
    merged = {}
    for p in list(PRIOR_RESULTS_PATHS) + [RESULTS_PATH]:
        part = load_json_safe(p)
        if part: print(f"  loaded {len(part)} record(s) from {p}")
        merged.update(part)
    return merged

def save_results(res):
    """Atomic: write a temp file, flush to disk, then rename over the real file (never leaves a half-written/empty JSON)."""
    tmp = RESULTS_PATH + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(res, fh, indent=1, default=_json_default)
        fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, RESULTS_PATH)
print(f"results file: {RESULTS_PATH}")
print(f"mode={'DRY_RUN' if DRY_RUN else 'FULL'} | subjects={SUBJECTS} | classes={NCLASS_LIST} | models={MODELS_TO_RUN} | "
      f"variants={VARIANTS} | seeds={SEEDS}")

In [ ]:
# ---------------- configuration of THIS notebook ----------------
import subprocess, sys
try:
    import onnxruntime as ort
except ImportError:                     # not preinstalled on Kaggle: needs Internet on (Settings, Internet)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"])
    import onnxruntime as ort
try:
    import mne
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mne"])
    import mne
mne.set_log_level("ERROR")
DO_LIMB   = True
DO_FINGER = True
LIMB_TARGETS = {"A": (7, "left_right"), "B": (2, "fists_feet")}   # flagship subjects of the paper: task -> (subject, replay name)
FINGER_SUBJECT = 1
# The finger recordings use the BioSemi 128-channel (ABC) layout. BioSemi's electrode-position file lists these electrodes at the 10-20 positions of C3, Cz and C4:
FINGER_TRACE_MAP = {"C3": "D19", "CZ": "A1", "C4": "B22"}
FINGER_WAVEFORM_DECIMALS = 2                      # keeps the finger logs a manageable size
LIMB_WAVEFORM_DECIMALS = 3
MIN_AGREEMENT = 0.999                             # share of windows on which ONNX must reproduce the stored prediction
REPLAY_CHANNELS = [8, 10, 12, 1, 5, 17]           # C3, Cz, C4, FC3, FC4, CPz in the PhysioNet 64-channel order (as in the export notebook)
REPLAY_CHANNEL_NAMES = ["C3", "CZ", "C4", "FC3", "FC4", "CPZ"]
BANDS = {"theta": (4, 8), "mu": (8, 13), "beta": (13, 30), "gamma_low": (30, 40)}
OUT_DIR = os.path.join(WORK_DIR, "signal_export"); os.makedirs(OUT_DIR, exist_ok=True)

def find_input(name):
    for root in ["/kaggle/input", "."]:
        for d, _, files in os.walk(root):
            if name in files:
                return os.path.join(d, name)
    return None
print("replay logs found:", sum(find_input(f"replay_{t}_{m}.json") is not None for t in ["left_right", "fists_feet"] for m in MODES),
      "of 4 limb |", sum(find_input(f"replay_finger_{k}_{m}.json") is not None for k in [2, 3] for m in MODES), "of 4 finger")

## 1. Limb tasks: data, run mapping and preprocessing (verbatim from the export notebook)

In [ ]:
import os
import time
import mne
from mne.datasets import eegbci

MNE_DATA_DIR = "/kaggle/working/mne_data"
os.makedirs(MNE_DATA_DIR, exist_ok=True)

def edf_path(subject_id, run, max_retries=4, backoff_seconds=10):
    last_error = None
    for attempt in range(max_retries):
        try:
            paths = eegbci.load_data(subject_id, [run], path=MNE_DATA_DIR, update_path=False, verbose=False)
            return paths[0]
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait = backoff_seconds * (attempt + 1)
                print(f"    S{subject_id:03d} run {run}: download failed ({type(e).__name__}), retrying in {wait}s...")
                time.sleep(wait)
    raise last_error

In [ ]:
LEFT_RIGHT_FIST_RUNS = {"ME": [3, 7, 11], "MI": [4, 8, 12]}
FISTS_FEET_RUNS = {"ME": [5, 9, 13], "MI": [6, 10, 14]}
LEFT_RIGHT_LABELS = {"T0": "rest", "T1": "left", "T2": "right"}
FISTS_FEET_LABELS = {"T0": "rest", "T1": "both_fists", "T2": "both_feet"}
KNOWN_BAD_SUBJECTS = [88, 92, 100, 104, 106]

In [ ]:
WINDOW_SEC, STEP_SEC = 1.0, 0.125
TRIAL_TMIN, TRIAL_TMAX = 0.0, 4.0

# Preprocessing ablations (defaults = the protocol behind every number reported so far). Set in the configuration cell.
NORM_MODE = "window"        # "window": z-score each channel inside every 1 s window (original)
                            # "run":    z-score each channel over the whole filtered run, so window-to-window power changes (ERD/ERS) survive
CHANNEL_SUBSET = None       # None = all 64 channels; or SENSORIMOTOR_CHANNELS (21 central channels, matched case-insensitively)
SENSORIMOTOR_CHANNELS = ["FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6", "C5", "C3", "C1", "Cz", "C2", "C4", "C6",
                         "CP5", "CP3", "CP1", "CPz", "CP2", "CP4", "CP6"]

def pick_channels(raw):
    if CHANNEL_SUBSET is None:
        return raw
    wanted = {c.lower() for c in CHANNEL_SUBSET}
    keep = [ch for ch in raw.ch_names if ch.lower() in wanted]
    if len(keep) != len(wanted):
        raise ValueError(f"channel subset: found {len(keep)} of {len(wanted)} requested channels")
    raw.pick(keep)
    return raw

def load_and_preprocess(path):
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    raw.rename_channels(lambda ch: ch.strip("."))
    raw.set_eeg_reference("average", projection=False, verbose=False)
    raw.filter(l_freq=4.0, h_freq=40.0, fir_design="firwin", verbose=False)
    raw = pick_channels(raw)                 # after the common average reference, which uses all 64 channels
    return raw

def epoch_run(raw, run_number, label_map):
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    class_event_id = {label_map[c]: eid for c, eid in event_id.items() if c in label_map}
    return mne.Epochs(raw, events, event_id=class_event_id, tmin=TRIAL_TMIN, tmax=TRIAL_TMAX,
                       baseline=None, preload=True, verbose=False)

def sliding_windows(epochs, sfreq, ref_stats=None):
    data = epochs.get_data()
    labels = epochs.events[:, -1]
    id_to_label = {v: k for k, v in epochs.event_id.items()}
    win_n, step_n = int(WINDOW_SEC * sfreq), int(STEP_SEC * sfreq)
    X, y, trial_ids = [], [], []
    for i in range(data.shape[0]):
        trial, label = data[i], id_to_label[labels[i]]
        start = 0
        while start + win_n <= trial.shape[1]:
            w = trial[:, start:start + win_n]
            if ref_stats is None:
                w = (w - w.mean(axis=1, keepdims=True)) / (w.std(axis=1, keepdims=True) + 1e-8)      # per-window (original)
            else:
                w = (w - ref_stats[0]) / (ref_stats[1] + 1e-8)                                         # per-run statistics
            X.append(w); y.append(label); trial_ids.append(i)
            start += step_n
    return np.stack(X), np.array(y), np.array(trial_ids)

def load_subjects(subject_ids, mode_runs, label_map, exclude_bad=True):
    if exclude_bad:
        subject_ids = [s for s in subject_ids if s not in KNOWN_BAD_SUBJECTS]
    Xs, ys, trial_id_chunks, mode_chunks = [], [], [], []
    trial_offset = 0
    for sid in subject_ids:
        for mode, runs in mode_runs.items():
            for run in runs:
                path = edf_path(sid, run)
                if not os.path.exists(path):
                    continue
                try:
                    raw = load_and_preprocess(path)
                    epochs = epoch_run(raw, run, label_map)
                    stats_run = None
                    if NORM_MODE == "run":
                        d = raw.get_data(); stats_run = (d.mean(axis=1, keepdims=True), d.std(axis=1, keepdims=True))
                    X, y, tids = sliding_windows(epochs, raw.info["sfreq"], stats_run)
                except Exception as e:
                    print(f"  skipping S{sid:03d} run {run}: {e}")
                    continue
                Xs.append(X); ys.append(y)
                trial_id_chunks.append(tids + trial_offset)
                mode_chunks.append(np.full(len(y), mode))
                trial_offset += tids.max() + 1 if len(tids) else 0
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(trial_id_chunks), np.concatenate(mode_chunks)

## 2. Split code (verbatim): rebuilds exactly the held-out windows of the exported models

In [ ]:
from sklearn.model_selection import train_test_split
SPLIT_MODE = "trial"
BASE_FRAC, FT_FRAC = 0.60, 0.20
LABEL_ORDER_LR = ["left", "rest", "right"]
LABEL_ORDER_FF = ["both_feet", "both_fists", "rest"]
TASK_CFG = {"A": dict(runs=LEFT_RIGHT_FIST_RUNS, labels=LEFT_RIGHT_LABELS, order=LABEL_ORDER_LR),
            "B": dict(runs=FISTS_FEET_RUNS, labels=FISTS_FEET_LABELS, order=LABEL_ORDER_FF)}

def split_by_trial(trial_ids, y_int, test_size, seed=0):
    """Boolean masks (train_mask, test_mask) such that every trial goes wholly to one side."""
    uniq = np.unique(trial_ids)
    lab = np.array([y_int[trial_ids == t][0] for t in uniq])
    try:
        tr, te = train_test_split(uniq, test_size=test_size, stratify=lab, random_state=seed)
    except ValueError:    # a class has too few trials to stratify
        tr, te = train_test_split(uniq, test_size=test_size, random_state=seed)
    return np.isin(trial_ids, tr), np.isin(trial_ids, te)


def plan_splits(y_int, trial_ids, modes, split_mode=None, seed=0):
    """Returns (base_train_idx, base_val_idx, {mode: {ft_train, ft_val, eval}}) as index arrays."""
    split_mode = split_mode or SPLIT_MODE
    base_parts, late = [], {}
    for mode in np.unique(modes):
        idx = np.where(modes == mode)[0]
        if split_mode == "trial":
            ordered = np.unique(trial_ids[idx])                  # ids grow with recording order
            n_base = max(1, int(round(len(ordered) * BASE_FRAC)))
            in_base = np.isin(trial_ids[idx], ordered[:n_base])
            base_idx, late_idx = idx[in_base], idx[~in_base]
        else:                                                     # legacy: first 70 % of windows
            cut = int(len(idx) * 0.7)
            base_idx, late_idx = idx[:cut], idx[cut:]
        base_parts.append(base_idx); late[mode] = late_idx
    base_idx = np.concatenate(base_parts)

    if split_mode == "trial":
        tr_m, va_m = split_by_trial(trial_ids[base_idx], y_int[base_idx], 0.15, seed)
        base_train, base_val = base_idx[tr_m], base_idx[va_m]
    else:
        shuffled = base_idx[np.random.RandomState(0).permutation(len(base_idx))]
        cut = int(len(shuffled) * 0.85)
        base_train, base_val = shuffled[:cut], shuffled[cut:]

    plans = {}
    for mode, late_idx in late.items():
        if len(late_idx) < 20:
            continue
        if split_mode == "trial":
            eval_share = (1 - BASE_FRAC - FT_FRAC) / (1 - BASE_FRAC)
            ft_m, ev_m = split_by_trial(trial_ids[late_idx], y_int[late_idx], eval_share, seed)
            ft_idx, eval_idx = late_idx[ft_m], late_idx[ev_m]
            a, b = split_by_trial(trial_ids[ft_idx], y_int[ft_idx], 0.2, seed)
            ft_train, ft_val = ft_idx[a], ft_idx[b]
        else:
            ft_idx, eval_idx = train_test_split(late_idx, test_size=0.5, stratify=y_int[late_idx], random_state=0)
            ft_train, ft_val = train_test_split(ft_idx, test_size=0.2, stratify=y_int[ft_idx], random_state=0)
        plans[mode] = {"ft_train": ft_train, "ft_val": ft_val, "eval": eval_idx}
    return base_train, base_val, plans

## 3. Signal helpers and the alignment checks

In [ ]:
class AlignmentError(RuntimeError):
    pass

def relative_band_power(window, fs):
    """window (channels, samples), already z-scored per channel as the model sees it.
    Returns {band: [relative power per channel]}: share of the 4-40 Hz power in each band.
    A Hann window is applied; the frequency resolution is fs / samples (1 Hz for a 1 s window)."""
    w = np.asarray(window, dtype=np.float64)
    w = w - w.mean(axis=1, keepdims=True)
    spec = np.abs(np.fft.rfft(w * np.hanning(w.shape[1]), axis=1)) ** 2
    freqs = np.fft.rfftfreq(w.shape[1], 1.0 / fs)
    total = spec[:, (freqs >= 4) & (freqs < 40)].sum(axis=1) + 1e-12
    out = {}
    for name, (lo, hi) in BANDS.items():
        sel = (freqs >= lo) & (freqs < hi)
        out[name] = [round(float(v), 4) for v in spec[:, sel].sum(axis=1) / total]
    return out

def _softmax(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def onnx_predictions(path, X, batch=128):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    out = [sess.run(None, {name: X[i:i + batch].astype(np.float32)})[0] for i in range(0, len(X), batch)]
    return _softmax(np.concatenate(out))

class CountMismatch(AlignmentError):
    pass

def _assemble(existing, Xrec, fs, ch_idx, ch_names, decimals, note, mapping=None):
    """Xrec[j] is the window that belongs to stored record j."""
    new = json.loads(json.dumps(existing))
    for r, w in zip(new["records"], Xrec):
        if len(ch_idx):
            r["waveform"] = [[round(float(v), decimals) for v in w[c]] for c in ch_idx]
            r["bands"] = relative_band_power(w[list(ch_idx)], fs)
        else:                                                    # no channel mapping: band power of the mean over all channels
            r.pop("waveform", None)
            r["bands"] = {k: v[:1] for k, v in relative_band_power(w.mean(axis=0, keepdims=True), fs).items()}
    new["channels_shown"] = [int(c) for c in ch_idx]; new["channel_names"] = list(ch_names)
    n_samp = int(Xrec.shape[2])
    sig = {
        "sampling_rate_hz": fs, "samples_per_window": n_samp,
        "traces": "full-rate samples of the displayed channels, as the model sees them (common average reference, 4-40 Hz band-pass, z-scored per channel inside each 1 s window)" if len(ch_idx) else "none stored",
        "band_power": "relative power: share of the 4-40 Hz power of the window, Hann window, " + ("per displayed channel" if len(ch_idx) else "of the mean over all channels"),
        "bands_hz": BANDS, "alignment": note}
    if mapping: sig["channel_mapping"] = mapping
    new["meta"]["signal"] = sig
    return new

def build_signal_doc(existing, Xe, ye_int, te, fs, ch_idx, ch_names, decimals, onnx_path=None, label="", mapping=None):
    """Returns a copy of the replay log with full-rate traces and band power added, or raises AlignmentError."""
    order = existing["label_order"]
    idx = np.argsort(np.asarray(te), kind="stable")             # same ordering rule as the export notebooks
    Xo, yo, to = Xe[idx], np.asarray(ye_int)[idx], np.asarray(te)[idx]
    recs = existing["records"]
    if len(recs) != len(Xo):
        raise CountMismatch(f"{label}: {len(Xo)} rebuilt windows but the stored log has {len(recs)}")
    if [int(r["trial"]) for r in recs] != [int(t) for t in to]:
        raise AlignmentError(f"{label}: trial ids differ from the stored log")
    if [r["true_class"] for r in recs] != [order[int(y)] for y in yo]:
        raise AlignmentError(f"{label}: true classes differ from the stored log")
    note = "checks 1-3 passed (window count, trial ids, true classes)"
    if onnx_path:
        p = onnx_predictions(onnx_path, Xo)
        pred = [order[int(k)] for k in p.argmax(1)]
        agree = float(np.mean([a == r["predicted_class"] for a, r in zip(pred, recs)]))
        dprob = float(np.mean(np.abs(p - np.array([[r["probs"][c] for c in order] for r in recs]))))
        if agree < MIN_AGREEMENT:
            raise AlignmentError(f"{label}: ONNX agrees with the stored predictions on only {agree:.4f} of the windows")
        note += f"; ONNX reproduces the stored predictions on {agree:.4f} of windows (mean |dp| = {dprob:.1e})"
    else:
        note += "; ONNX file not found, check 4 skipped"
    new = _assemble(existing, Xo, fs, ch_idx, ch_names, decimals, note, mapping)
    print(f"  {label}: written ({note})")
    return new

def build_signal_doc_by_probs(existing, Xe, ye_int, te, fs, ch_idx, ch_names, decimals, onnx_path, label="", mapping=None, tol=1e-4):
    """Used when the number of rebuilt windows differs from the stored log. Every stored window is matched to a rebuilt
    window of the SAME trial by its ONNX probabilities (agreement within tol on every class), keeping the stored order.
    A window is only given a signal if the exported model produces exactly its stored probabilities on it."""
    if not onnx_path:
        raise AlignmentError(f"{label}: counts differ and no ONNX file is available to match windows")
    order = existing["label_order"]
    idx = np.argsort(np.asarray(te), kind="stable")
    Xo, yo, to = Xe[idx], np.asarray(ye_int)[idx], np.asarray(te)[idx]
    P = onnx_predictions(onnx_path, Xo)
    recs = existing["records"]
    S = np.array([[r["probs"][c] for c in order] for r in recs])
    by = {}
    for i, t in enumerate(to): by.setdefault(int(t), []).append(i)
    rn = np.array([len(v) for v in by.values()]); sn = np.array(list(__import__("collections").Counter(int(r["trial"]) for r in recs).values()))
    print(f"  windows per trial: rebuilt {rn.min()}-{rn.max()} (mean {rn.mean():.1f}), stored {sn.min()}-{sn.max()} (mean {sn.mean():.1f})")
    chosen, contiguous, greedy, worst = [], 0, 0, 0.0
    j = 0
    while j < len(recs):
        t = int(recs[j]["trial"]); k = j
        while k < len(recs) and int(recs[k]["trial"]) == t: k += 1
        R = by.get(t)
        if R is None:
            raise AlignmentError(f"{label}: trial {t} of the stored log does not exist among the rebuilt windows")
        ns = k - j; best_o, best_d = None, 1e9
        for o in range(0, len(R) - ns + 1):
            d = float(np.abs(S[j:k] - P[R[o:o + ns]]).max())
            if d < best_d: best_o, best_d = o, d
        if best_o is not None and best_d < tol:
            sel = R[best_o:best_o + ns]; contiguous += 1; worst = max(worst, best_d)
        else:                                                    # windows missing inside the trial: ordered one-to-one matching
            sel, ptr = [], 0
            for jj in range(j, k):
                found = None
                while ptr < len(R):
                    d = float(np.abs(S[jj] - P[R[ptr]]).max()); ptr += 1
                    if d < tol: found = R[ptr - 1]; worst = max(worst, d); break
                if found is None:
                    raise AlignmentError(f"{label}: stored window {jj} (trial {t}) has no rebuilt window with the same ONNX probabilities")
                sel.append(found)
            greedy += 1
        chosen.extend(sel); j = k
    if [order[int(yo[i])] for i in chosen] != [r["true_class"] for r in recs]:
        raise AlignmentError(f"{label}: matched windows have different true classes than the stored log")
    note = (f"matched by ONNX probabilities, not by count: {len(Xo)} rebuilt vs {len(recs)} stored windows; "
            f"{contiguous} trials as one contiguous block, {greedy} trials with gaps; largest probability difference {worst:.1e}")
    new = _assemble(existing, Xo[chosen], fs, ch_idx, ch_names, decimals, note, mapping)
    print(f"  {label}: written ({note})")
    return new

def write_doc(doc, name):
    with open(os.path.join(OUT_DIR, name), "w") as fh:
        json.dump(doc, fh)

# ---- self-test on synthetic data: a 10 Hz wave must put its power into the mu band ----
_t = np.arange(160) / 160.0
_bp = relative_band_power(np.stack([np.sin(2 * np.pi * 10 * _t), np.sin(2 * np.pi * 20 * _t)]), 160)
assert _bp["mu"][0] > 0.8 and _bp["beta"][1] > 0.8, _bp
print("band-power self-test passed")

## 4. Run: limb tasks (loads two subjects, no training)

In [ ]:
SIGNAL_LOGS = {}      # name -> status, printed at the end
if DO_LIMB:
    for task, (sid, tname) in LIMB_TARGETS.items():
        cfg = TASK_CFG[task]; order = cfg["order"]
        print(f"\n=== Limb task {task} ({tname}), subject S{sid:03d} ===")
        X, y, tid, modes = load_subjects([sid], cfg["runs"], cfg["labels"], exclude_bad=False)
        y_int = np.array([order.index(l) for l in y])
        _, _, plans = plan_splits(y_int, tid, modes)
        for mode, pl in plans.items():
            name = f"replay_{tname}_{mode}.json"
            path = find_input(name)
            if path is None:
                SIGNAL_LOGS[name] = "SKIPPED: stored replay log not found in /kaggle/input"; print("  " + SIGNAL_LOGS[name]); continue
            existing = json.load(open(path))
            ev = pl["eval"]
            onnx = find_input(f"tcformer-{task}-s{sid:03d}-{mode.lower()}.onnx")
            try:
                doc = build_signal_doc(existing, X[ev], y_int[ev], tid[ev], 160, REPLAY_CHANNELS, REPLAY_CHANNEL_NAMES,
                                       LIMB_WAVEFORM_DECIMALS, onnx, label=f"{tname} {mode}")
                write_doc(doc, name); SIGNAL_LOGS[name] = "written"
            except AlignmentError as e:
                SIGNAL_LOGS[name] = f"NOT WRITTEN: {e}"; print("  " + SIGNAL_LOGS[name])
        del X

## 5. Finger task: download, loader and plan (verbatim from the finger export notebook)

In [ ]:
import requests

def list_figshare_files(article_id=FIGSHARE_ID, show=True):
    files, page = [], 1
    while True:
        r = requests.get(f"https://api.figshare.com/v2/articles/{article_id}/files",
                         params={"page": page, "page_size": 100}, timeout=60)
        r.raise_for_status()
        chunk = r.json()
        if not chunk:
            break
        files += chunk; page += 1
    if show:
        total = sum(f["size"] for f in files)
        print(f"{len(files)} files, {total / 1e9:.1f} GB total")
        for f in sorted(files, key=lambda f: f["name"]):
            print(f"  {f['size'] / 1e9:8.2f} GB  {f['name']}")
    return files

def files_for_subject(files, subj):
    if subj in SUBJECT_FILE_MAP:
        wanted = set(SUBJECT_FILE_MAP[subj]); return [f for f in files if f["name"] in wanted]
    pat = re.compile(rf"(^|[^A-Za-z0-9])S(ubject)?[_ -]?0*{subj}([^0-9]|$)", re.I)
    return [f for f in files if pat.search(f["name"])]

def download(url, dest, expected_size=None):
    done, t0 = 0, time.time()
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                fh.write(chunk); done += len(chunk)
                if int((time.time() - t0)) % 60 == 0 and done % (64 * 1024 * 1024) < 8 * 1024 * 1024:
                    print(f"    {done / 1e9:.2f} GB downloaded", end="\r")
    if expected_size and done != expected_size:
        raise IOError(f"incomplete download: {done} of {expected_size} bytes")

def fetch_subject_raw(subj):
    """Returns (folder_with_raw_data, cleanup_fn)."""
    if DATA_ROOT:
        return DATA_ROOT, (lambda: None)
    files = files_for_subject(list_figshare_files(show=False), subj)
    if not files:
        raise FileNotFoundError(f"No Figshare file matched subject {subj}. Run list_figshare_files() and fill SUBJECT_FILE_MAP.")
    need = sum(f["size"] for f in files) * 2.2          # archive + extracted copy
    free = shutil.disk_usage(TMP_DIR).free
    if free < need:
        raise OSError(f"Subject {subj} needs ~{need / 1e9:.0f} GB scratch but only {free / 1e9:.0f} GB is free in {TMP_DIR}. "
                      "Use a larger-disk environment, or attach pre-extracted data and set DATA_ROOT.")
    out = os.path.join(TMP_DIR, f"finger_S{subj:02d}")
    shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
    for f in files:
        dest = os.path.join(out, f["name"])
        print(f"  downloading {f['name']} ({f['size'] / 1e9:.2f} GB)")
        for attempt in range(3):
            try:
                download(f["download_url"], dest, f["size"]); break
            except Exception as e:
                print(f"    download attempt {attempt + 1}/3 failed ({type(e).__name__}: {e})")
                if os.path.exists(dest): os.remove(dest)
                if attempt == 2: raise
                time.sleep(30 * (attempt + 1))
        if zipfile.is_zipfile(dest):
            with zipfile.ZipFile(dest) as z: z.extractall(out)
            os.remove(dest)
        elif tarfile.is_tarfile(dest):
            with tarfile.open(dest) as t: t.extractall(out)
            os.remove(dest)
    return out, (lambda: shutil.rmtree(out, ignore_errors=True))

In [ ]:
def load_mat_trials(path, maxtriallen=5):
    """-> trials (n, n_chan, maxtriallen*srate) NaN-padded, labels (n,), srate. CAR is applied here, as in the official code."""
    mat = scipy.io.loadmat(path)
    eeg, event = mat["eeg"], mat["event"]
    signals = eeg["data"][0][0]
    srate = float(np.squeeze(eeg["fsample"][0][0]))
    starts, ends, targets = [], [], []
    for i in range(event.shape[1]):
        evt = event[0, i]
        etype = str(np.squeeze(evt["type"]))
        sample = int(np.squeeze(evt["sample"]))
        value = np.squeeze(evt["value"])
        if etype == "Target":
            starts.append(sample - 1); targets.append(int(value))
        elif etype == "TrialEnd":
            ends.append(sample - 1)
    n_max = int(maxtriallen * srate)
    n = min(len(starts), len(ends))
    trials = np.full((n, signals.shape[0], n_max), np.nan, dtype=np.float32)
    for k in range(n):
        seg = signals[:, starts[k]:ends[k]][:, :n_max]
        trials[k, :, :seg.shape[1]] = seg
    trials = trials - trials.mean(axis=1, keepdims=True)          # common average reference
    return trials, np.array(targets[:n]), srate


def preprocess_trials(trials, srate, p=PARAMS, zero_phase=ZERO_PHASE):
    """-> windows (m, n_chan, 100) float32, trial_index (m,). Windows touching the NaN padding are dropped."""
    seg = int(p["windowlen"] * srate); step = int(p["step_sec"] * srate)
    desired = int(p["windowlen"] * p["downsrate"])
    b, a = scipy.signal.butter(4, list(p["bandpass"]), btype="bandpass", fs=p["downsrate"])
    W, T = [], []
    for ti in range(trials.shape[0]):
        tr = trials[ti]
        starts = [s for s in range(0, tr.shape[1] - seg + 1, step) if not np.isnan(tr[:, s:s + seg]).any()]
        if not starts:
            continue
        x = np.stack([tr[:, s:s + seg] for s in starts])                    # (k, chan, seg)
        x = scipy.signal.resample(x, desired, axis=2)                        # 1024 Hz -> 100 Hz
        if zero_phase:
            x = scipy.signal.filtfilt(b, a, x, axis=-1)
        else:
            pad = 100
            xp = np.pad(x, ((0, 0), (0, 0), (pad, pad)))
            x = scipy.signal.lfilter(b, a, xp, axis=-1)[:, :, pad:-pad]
        x = (x - x.mean(axis=2, keepdims=True)) / (x.std(axis=2, keepdims=True) + 1e-8)   # z-score per channel
        W.append(x.astype(np.float32)); T.extend([ti] * len(x))
    if not W:
        return np.zeros((0, trials.shape[1], desired), np.float32), np.zeros(0, int)
    return np.concatenate(W), np.array(T)


def describe_folder(folder):
    """mode / session / class-setting parsed from a data-folder name (names follow the official glob patterns)."""
    low = folder.lower()
    mode = "ME" if "movement" in low else "MI" if "imagery" in low else None
    m = re.search(r"sess(\d+)", low)
    return mode, (int(m.group(1)) if m else 0), ("3class" if "3class" in low else "2class" if "2class" in low else "")


def _natkey(s):
    """Natural sort key: 'Run2' < 'Run10' (plain text sorting would put Run10 first and scramble the chronology)."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(s))]


def discover_mat_files(root, subj):
    base = os.path.join(root, f"S{subj:02d}")
    if not os.path.isdir(base):
        cands = [d for d, _, _ in os.walk(root) if os.path.basename(d).upper() == f"S{subj:02d}"]
        base = cands[0] if cands else root
    found = []
    for dirpath, _, fnames in os.walk(base):
        mats = sorted((f for f in fnames if f.lower().endswith(".mat")), key=_natkey)
        if not mats:
            continue
        mode, sess, cls = describe_folder(os.path.basename(dirpath))
        if mode is None:
            continue
        for f in mats:
            found.append(dict(path=os.path.join(dirpath, f), mode=mode, session=sess, folder=os.path.basename(dirpath)))
    found.sort(key=lambda d: (d["mode"], d["session"], _natkey(d["folder"]), _natkey(os.path.basename(d["path"]))))
    if not found:
        raise FileNotFoundError(f"No .mat files under folders named *Movement*/*Imagery* in {base}. Check the extracted layout.")
    return found


def cache_subject(subj, force=False, offline_only=False, stride=1):
    tag = f"_offline_s{stride}" if offline_only else ""
    path = os.path.join(CACHE_DIR, f"S{subj:02d}_v21{tag}.npz")
    if os.path.exists(path) and not force:
        return path
    root, cleanup = fetch_subject_raw(subj)
    try:
        X, Y, TID, MODE, SESS, FOLD = [], [], [], [], [], []
        uid = 0
        for d in discover_mat_files(root, subj):
            if offline_only and d["session"] != 0:
                continue
            trials, labels, srate = load_mat_trials(d["path"], PARAMS["maxtriallen"])
            w, ti = preprocess_trials(trials, srate)
            if len(w) == 0:
                continue
            keep = np.arange(0, len(w), stride)
            w, ti = w[keep], ti[keep]
            X.append(w.astype(np.float16)); Y.append(labels[ti]); TID.append(ti + uid)
            MODE.append(np.full(len(w), d["mode"])); SESS.append(np.full(len(w), d["session"]))
            FOLD.append(np.full(len(w), d["folder"]))
            uid += trials.shape[0]
            print(f"  S{subj:02d} {d['mode']} sess{d['session']} {d['folder']}/{os.path.basename(d['path'])}: {trials.shape[0]} trials, {len(w)} windows")
        np.savez_compressed(path, X=np.concatenate(X), y=np.concatenate(Y), trial=np.concatenate(TID),
                            mode=np.concatenate(MODE), session=np.concatenate(SESS), folder=np.concatenate(FOLD))
    finally:
        cleanup()
    return path

def load_cache(path):
    with np.load(path) as f:
        return {k: f[k] for k in f.files}

In [ ]:
def plan_finger(z, nclass):
    keep = CLASS_SETS[nclass]; lab_to_int = {l: i for i, l in enumerate(keep)}
    sess_all, fol_all = z["session"], z["folder"].astype(str)
    sel = np.isin(z["y"], keep)
    if nclass == 3:
        sel &= (sess_all == 0) | (np.char.find(fol_all, "3class") >= 0)
    elif TWO_CLASS_FOLDERS == "2class":
        sel &= (sess_all == 0) | (np.char.find(fol_all, "2class") >= 0)
    idx = np.where(sel)[0]
    X = z["X"][idx]; trial = z["trial"][idx]; mode = z["mode"][idx]; sess = sess_all[idx]
    y = np.array([lab_to_int[int(l)] for l in z["y"][idx]])
    test_sess = {m: int(sess[mode == m].max()) for m in MODES
                 if (mode == m).any() and len(set(sess[mode == m].tolist())) >= 2}
    assert test_sess, "no mode has an offline session plus at least one later session"
    base_mask = np.zeros(len(y), bool)
    for m, ts in test_sess.items():
        base_mask |= (mode == m) & (sess < ts)
    idx_base = np.where(base_mask)[0]
    tr_m, va_m = split_by_trial(trial[idx_base], y[idx_base], 0.15)
    plan = dict(X=X, y=y, trial=trial, mode=mode, sess=sess, nclass=nclass, label_order=[f"label{l}" for l in keep],
                base_train=idx_base[tr_m], base_val=idx_base[va_m], test_sess=test_sess, modes={})
    for m, ts in test_sess.items():
        idm = np.where((mode == m) & (sess == ts))[0]
        order = np.unique(trial[idm])
        n_ft = max(1, int(round(len(order) * FT_FRAC)))
        ft_idx = idm[np.isin(trial[idm], order[:n_ft])]
        ev_idx = idm[np.isin(trial[idm], order[n_ft:])]
        if len(np.unique(trial[ev_idx])) < 3:
            print(f"  {m}: skipped (fewer than 3 evaluation trials)"); continue
        a, b = split_by_trial(trial[ft_idx], y[ft_idx], 0.2)
        ft_train, ft_val = ft_idx[a], ft_idx[b]
        if len(ft_val) == 0:
            ft_val = ft_train
        if len(np.unique(y[ft_train])) < nclass:
            print(f"  {m}: skipped (a class is missing from the fine-tune trials)"); continue
        plan["modes"][m] = dict(ft_train=ft_train, ft_val=ft_val, eval=ev_idx)
    return plan

def verify_finger_plan(plan, label):
    T = lambda idx: set(plan["trial"][idx].tolist())
    nclass = plan["nclass"]
    assert len(plan["modes"]) > 0, f"{label}: no mode can be evaluated"
    assert len(np.unique(plan["y"][plan["base_train"]])) == nclass, f"{label}: a class is missing from the pretraining data"
    assert not (T(plan["base_train"]) & T(plan["base_val"])), f"{label}: pretrain and its validation share a trial"
    for m, p in plan["modes"].items():
        parts = {"pretrain": T(plan["base_train"]), "pretrain_val": T(plan["base_val"]),
                 "ft_train": T(p["ft_train"]), "ft_val": T(p["ft_val"]), "eval": T(p["eval"])}
        names = list(parts)
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                if names[i] == "ft_train" and names[j] == "ft_val" and p["ft_val"] is p["ft_train"]:
                    continue
                shared = parts[names[i]] & parts[names[j]]
                assert not shared, f"{label} {m}: LEAKAGE - {len(shared)} trial(s) in both {names[i]} and {names[j]}"
        ts = plan["test_sess"][m]
        assert (plan["sess"][p["eval"]] == ts).all() and (plan["sess"][p["ft_train"]] == ts).all(), f"{label} {m}: wrong session"
        assert (plan["sess"][plan["base_train"]][plan["mode"][plan["base_train"]] == m] < ts).all(), f"{label} {m}: pretraining includes the test session"
        assert plan["trial"][p["ft_train"]].max() < plan["trial"][p["eval"]].min(), f"{label} {m}: evaluation is not strictly later than fine-tuning"
        assert (plan["mode"][p["eval"]] == m).all(), f"{label} {m}: evaluation windows from another mode"
    return {m: len(T(p["eval"])) for m, p in plan["modes"].items()}

In [ ]:
# Capture the channel labels of the .mat files the first time a file is read (needed to name the displayed channels).
LABELS_FILE = os.path.join(WORK_DIR, "finger_channel_labels.json")
_orig_load_mat_trials = load_mat_trials
def load_mat_trials(path, maxtriallen=5):
    if not os.path.exists(LABELS_FILE):
        try:
            mat = scipy.io.loadmat(path, variable_names=["eeg"])
            lab = [str(np.squeeze(x)) for x in np.ravel(mat["eeg"]["label"][0][0])]
            with open(LABELS_FILE, "w") as fh: json.dump(lab, fh)
            print(f"  channel labels captured: {len(lab)} (first ten: {lab[:10]})")
        except Exception as exc:
            print(f"  could not read channel labels ({type(exc).__name__}: {exc})")
    return _orig_load_mat_trials(path, maxtriallen)

## 6. Run: finger task (downloads S01, no training)

In [ ]:
if DO_FINGER:
    print(f"\n=== Finger task, subject S{FINGER_SUBJECT:02d} ===")
    zd = load_cache(cache_subject(FINGER_SUBJECT))
    labels = json.load(open(LABELS_FILE)) if os.path.exists(LABELS_FILE) else None
    if labels is None:                  # the labels captured from S01 in an earlier run were exactly the standard BioSemi order A1..A32, B1..B32, C1..C32, D1..D32
        labels = [f"{b}{i}" for b in "ABCD" for i in range(1, 33)]
        print("  channel labels not captured in this run: using the standard BioSemi 128 order (identical to the labels read from S01 earlier)")
    ch_idx, ch_names, mapping = [], [], None
    if labels and all(v in labels for v in FINGER_TRACE_MAP.values()):
        ch_names = list(FINGER_TRACE_MAP); ch_idx = [labels.index(FINGER_TRACE_MAP[c]) for c in ch_names]
        mapping = {c: f"{FINGER_TRACE_MAP[c]} (listed by BioSemi as coinciding with this 10-20 position)" for c in ch_names}
        print("  trace channels:", mapping)
    if not ch_idx:
        print("  BioSemi labels not found (or labels unavailable): the finger logs get band power of the mean over all channels and NO traces.")
        if labels: print("  labels start with:", labels[:12], "...")
    for k in NCLASS_LIST:
        plan = plan_finger(zd, k); verify_finger_plan(plan, f"S{FINGER_SUBJECT:02d} {k}-finger")
        for m, p in plan["modes"].items():
            name = f"replay_finger_{k}_{m}.json"
            path = find_input(name)
            if path is None:
                SIGNAL_LOGS[name] = "SKIPPED: stored replay log not found in /kaggle/input"; print("  " + SIGNAL_LOGS[name]); continue
            existing = json.load(open(path))
            onnx = find_input(f"tcformer-finger{k}-s{FINGER_SUBJECT:02d}-{m.lower()}.onnx")
            ev = p["eval"]
            try:
                args = (existing, plan["X"][ev].astype(np.float32), plan["y"][ev], plan["trial"][ev], 100, ch_idx, ch_names, FINGER_WAVEFORM_DECIMALS)
                try:
                    doc = build_signal_doc(*args, onnx, label=f"finger {k} {m}", mapping=mapping)
                except CountMismatch as e:
                    print("  " + str(e) + " -> matching windows by ONNX probabilities instead")
                    doc = build_signal_doc_by_probs(*args, onnx, label=f"finger {k} {m}", mapping=mapping)
                write_doc(doc, name); SIGNAL_LOGS[name] = "written"
            except AlignmentError as e:
                SIGNAL_LOGS[name] = f"NOT WRITTEN: {e}"; print("  " + SIGNAL_LOGS[name])
        del plan

## 7. Summary and download

In [ ]:
import zipfile
print("\nSTATUS OF EVERY LOG")
for k, v in sorted(SIGNAL_LOGS.items()): print(f"  {k:32s} {v}")
written = [k for k, v in SIGNAL_LOGS.items() if v == "written"]
zp = os.path.join(WORK_DIR, "neurobot_signal_logs.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for k in written: z.write(os.path.join(OUT_DIR, k), k)
print(f"\n{len(written)} log(s) in {zp} ({os.path.getsize(zp) / 1e6:.1f} MB)")
if len(written) < len(SIGNAL_LOGS) or not written:
    print("Some logs were NOT written. Read the messages above: a failed alignment check means the rebuilt windows differ from the stored ones, so nothing was attached.")

## After the run

1. Download **`neurobot_signal_logs.zip`** from the Output tab.
2. Copy the `replay_*.json` files from it over the files with the same names in `backend/data/` of the repository. Logs that were not written keep their old version, so a partial result is safe to use.
3. Run the backend tests (`pytest backend/tests`), commit, and let Vercel and the Space redeploy. The console then shows the real traces and the real relative band power, and the paper's Signal-tab text can be updated.

Nothing here changes a model, an accuracy figure or a table of the paper.